In [1]:
import numpy as np

POP = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25]
])

In [3]:
print(POP[1,1])

7


In [2]:
base_HV

array([0.2 , 0.25, 0.3 , 0.35, 0.4 ])

In [2]:
import numpy as np
hl = np.zeros(32)    # Alturas dos painéis
pa = np.zeros(32)    # Força de protensão
apl = np.zeros(32)   # Área de protensão

for i in range(32):
    if i < 4:
        hl[i] = 0.09
        pa[i] = -0.184350828 * (i + 1)
        apl[i] = 0.0001308 * (i + 1)
    elif i < 9:
        hl[i] = 0.13
        pa[i] = -0.245801104 * (i + 1)
        apl[i] = 0.0001744 * (i + 1)
    elif i < 15:
        hl[i] = 0.17
        pa[i] = -0.30725138 * (i + 1)
        apl[i] = 0.000218 * (i + 1)
    elif i < 21:
        hl[i] = 0.2
        pa[i] = -0.368701656 * (i + 1)
        apl[i] = 0.0002616 * (i + 1)
    elif i < 27:
        hl[i] = 0.21
        pa[i] = -0.431843 * (i + 1)
        apl[i] = 0.000306 * (i + 1)
    else:
        hl[i] = 0.26
        pa[i] = -0.52430052 * (i + 1)
        apl[i] = 0.000372 * (i + 1)


In [3]:
for i in pa:
    print(i)

-0.184350828
-0.368701656
-0.553052484
-0.737403312
-1.2290055199999999
-1.474806624
-1.720607728
-1.966408832
-2.212209936
-3.0725138000000003
-3.37976518
-3.68701656
-3.9942679400000003
-4.301519320000001
-4.6087707
-5.899226496
-6.267928152
-6.6366298079999995
-7.005331464
-7.37403312
-7.742734776
-9.500546
-9.932388999999999
-10.364232
-10.796075
-11.227917999999999
-11.659761
-14.680414560000001
-15.204715080000001
-15.729015600000002
-16.25331612
-16.77761664


In [ ]:
class Node:
    """Representa um nó no gráfico."""

    node_count = 0
    node_restrictions = []

    def __init__(self, x: float, y: float, restriction_x: int, restriction_y: int, restriction_momentum: int):
        """
        Inicializa um novo nó.

        Args:
            x (float): A coordenada x do nó.
            y (float): A coordenada y do nó.
        """
        Node.node_count += 1
        self.x = x
        self.y = y
        self.restriction_x = restriction_x
        self.restriction_y = restriction_y
        self.restriction_momentum = restriction_momentum
        self.node_id = Node.node_count
        Node.node_restrictions.append(restriction_x)
        Node.node_restrictions.append(restriction_y)
        Node.node_restrictions.append(restriction_momentum)


class Element:
    """Representa um elemento que conecta dois nós."""

    element_count = 0

    def __init__(self, node_start: Node, node_end: Node, area: float, moment_of_inertia: float, Young_s_Modulus: float):
        """
        Inicializa um novo elemento.

        Args:
            node_start (Node): O nó inicial do elemento.
            node_end (Node): O nó final do elemento.
            area (float): A área transversal do elemento [cm2].
            moment_of_inertia (float): O momento de inércia do elemento [cm4].
            Young_s_Modulus (float): O módulo de elasticidade de Young do elemento[GPa].
        """
        Element.element_count += 1
        self.node_start = node_start
        self.node_end = node_end
        self.area = area
        self.moment_of_inertia = moment_of_inertia
        self.Young_s_Modulus = Young_s_Modulus
        self.element_id = Element.element_count
        self.length = self.calculate_length()
        self.sen = self.calculate_sen()
        self.cos = self.calculate_cos()
        self.rotation_matrix = self.calculate_rotation_matrix()
        self.local_stiffness_matrix = self.calculate_local_stiffness_matrix()
        self.global_stiffness_matrix = self.calculate_global_stiffness_matrix()

    def calculate_length(self):
        """
        Calcula o comprimento do elemento.
        """
        dx = self.node_end.x - self.node_start.x
        dy = self.node_end.y - self.node_start.y
        return (dx**2 + dy**2) ** 0.5

    def calculate_sen(self):
        """
        Calcula o seno do ângulo do elemento.
        """
        dx = self.node_end.x - self.node_start.x
        dy = self.node_end.y - self.node_start.y
        return dy / self.length

    def calculate_cos(self):
        """
        Calcula o cosseno do ângulo do elemento.
        """
        dx = self.node_end.x - self.node_start.x
        dy = self.node_end.y - self.node_start.y
        return dx / self.length

    def calculate_rotation_matrix(self):
        return np.array(
            [
                [self.cos, self.sen, 0, 0, 0, 0],
                [-self.sen, self.cos, 0, 0, 0, 0],
                [0, 0, 1, 0, 0, 0],
                [0, 0, 0, self.cos, self.sen, 0],
                [0, 0, 0, -self.sen, self.cos, 0],
                [0, 0, 0, 0, 0, 1],
            ]
        )

    def calculate_local_stiffness_matrix(self):
        """Calcula a matriz de rigidez local do elemento."""
        EA = self.Young_s_Modulus * self.area
        EI = self.Young_s_Modulus * self.moment_of_inertia

        k11 = EA / self.length
        k12 = 0
        k13 = 0
        k14 = -EA / self.length
        k15 = 0
        k16 = 0

        k21 = 0
        k22 = (12 * EI / (self.length**3)) / 10000
        k23 = (6 * EI / (self.length**2)) / 100
        k24 = 0
        k25 = -(12 * EI / (self.length**3)) / 10000
        k26 = (6 * EI / (self.length**2)) / 100

        k31 = 0
        k32 = (6 * EI / (self.length**2)) / 100
        k33 = (4 * EI / (self.length))
        k34 = 0
        k35 = -(6 * EI / (self.length**2)) / 100
        k36 = (2 * EI / (self.length))

        k41 = -EA / (self.length)
        k42 = 0
        k43 = 0
        k44 = EA / (self.length)
        k45 = 0
        k46 = 0

        k51 = 0
        k52 = -(12 * EI / (self.length**3)) / 10000
        k53 = -(6 * EI / (self.length**2)) / 100
        k54 = 0
        k55 = (12 * EI / (self.length**3)) / 10000
        k56 = -(6 * EI / (self.length**2)) / 100

        k61 = 0
        k62 = (6 * EI / (self.length**2)) / 100
        k63 = (2 * EI / (self.length))
        k64 = 0
        k65 = -(6 * EI / (self.length**2)) / 100
        k66 = (4 * EI / (self.length))

        return np.array(
            [
                [k11, k12, k13, k14, k15, k16],
                [k21, k22, k23, k24, k25, k26],
                [k31, k32, k33, k34, k35, k36],
                [k41, k42, k43, k44, k45, k46],
                [k51, k52, k53, k54, k55, k56],
                [k61, k62, k63, k64, k65, k66],
            ]
        )

    def calculate_global_stiffness_matrix(self):
        return np.transpose(self.rotation_matrix) @ self.local_stiffness_matrix @ self.rotation_matrix


"""# Criando nós
node1 = Node(0, 0, 0, 0, 0)
node2 = Node(1, 1, 0, 0, 0)
node3 = Node(2, 2, 0, 0, 0)

# Criando elementos
element1 = Element(node1, node2, 1.0, 2.0, 3.0)
element2 = Element(node2, node3, 1.5, 2.5, 3.5)

# Calculando seno e cosseno para os elementos
sen_element1 = element1.calculate_sen()
cos_element1 = element1.calculate_cos()
sen_element2 = element2.calculate_sen()
cos_element2 = element2.calculate_cos()

# Exibindo os resultados
print("Elemento 1:")
print("Seno do ângulo:", sen_element1)
print("Cosseno do ângulo:", cos_element1)

print("\nElemento 2:")
print("Seno do ângulo:", sen_element2)
print("Cosseno do ângulo:", cos_element2)
"""


In [238]:
# [0] Numero no
# [1] X no
# [2] Y no
AREA = 84
I = 1008
E = 200
no_1 = Node(0, 6, 0, 1, 0)
no_2 = Node(6, 6, 0, 2, 0)
no_3 = Node(0, 0, 0, 3, 0)
no_4 = Node(6, 0, 0, 4, 0)
nos = []
nos.append(no_1)
nos.append(no_2)
nos.append(no_3)
nos.append(no_4)


elemento1 = Element(no_1, no_2, AREA, I, E)
elemento2 = Element(no_3, no_1, AREA, I, E)
elemento3 = Element(no_4, no_2, AREA, I, E)

elementos = []
elementos.append(elemento1)
elementos.append(elemento2)
elementos.append(elemento3)


In [239]:
def matriz_rigidez_global(elementos):
  length_matriz = len(Node.node_restrictions)
  global_matrix = np.zeros((length_matriz,length_matriz))   
  for a in range(len(elementos)):
    coordenada_inicial_global = 3*elementos[a].node_start.node_id
    coordenada_final_global=3*elementos[a].node_end.node_id
    posicao_global = [coordenada_inicial_global-2,
    coordenada_inicial_global-1,
    coordenada_inicial_global,
    coordenada_final_global-2,
    coordenada_final_global-1,
    coordenada_final_global]
    for k in range(6):
      for l in range(6):
        i = posicao_global[k]
        j = posicao_global[l]
        global_matrix[i-1][j-1]+=elementos[a].global_stiffness_matrix[k][l]
  return global_matrix

global_matrix = matriz_rigidez_global(elementos)


In [240]:
iterar = global_matrix
for i in iterar:
    print()
    for j in i:
        print(f"{j:<8}", end=" ")



2801.12  0.0      336.0    -2800.0  0.0      0.0      -1.12    0.0      336.0    0.0      0.0      0.0      
0.0      2801.12  336.0    0.0      -1.12    336.0    0.0      -2800.0  0.0      0.0      0.0      0.0      
336.0    336.0    268800.0 0.0      -336.0   67200.0  -336.0   0.0      67200.0  0.0      0.0      0.0      
-2800.0  0.0      0.0      2801.12  0.0      336.0    0.0      0.0      0.0      -1.12    0.0      336.0    
0.0      -1.12    -336.0   0.0      2801.12  -336.0   0.0      0.0      0.0      0.0      -2800.0  0.0      
0.0      336.0    67200.0  336.0    -336.0   268800.0 0.0      0.0      0.0      -336.0   0.0      67200.0  
-1.12    0.0      -336.0   0.0      0.0      0.0      1.12     0.0      -336.0   0.0      0.0      0.0      
0.0      -2800.0  0.0      0.0      0.0      0.0      0.0      2800.0   0.0      0.0      0.0      0.0      
336.0    0.0      67200.0  0.0      0.0      0.0      -336.0   0.0      134400.0 0.0      0.0      0.0      
0.0      0.0      

In [ ]:
#Cargas 

In [3]:
for i in range(4):
  print(i)

for i in range(4,9):
  print(i)
for i in range(9,15):
  print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
